# **Import Modules**

In [16]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# **Load The Dataset**

In [3]:
train = pd.read_csv('train.csv')

test = pd.read_csv('test.csv')

In [4]:
train.head()

,id,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,0,e,8.80,f,s,u,f,a,c,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,a
1,1,p,4.51,x,h,o,f,a,c,n,...,NaN,y,o,NaN,NaN,t,z,NaN,d,w
2,2,e,6.94,f,s,b,f,x,c,w,...,NaN,s,n,NaN,NaN,f,f,NaN,l,w
3,3,e,3.88,f,y,g,f,s,NaN,g,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,u
4,4,e,5.85,x,l,w,f,d,NaN,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,g,a


# **Data Understanding**

In [5]:
test.head()

,id,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,3116945,8.64,x,NaN,n,t,NaN,NaN,w,11.13,...,b,NaN,w,u,w,t,g,NaN,d,a
1,3116946,6.90,o,t,o,f,NaN,c,y,1.27,...,NaN,NaN,n,NaN,NaN,f,f,NaN,d,a
2,3116947,2.00,b,g,n,f,NaN,c,n,6.18,...,NaN,NaN,n,NaN,NaN,f,f,NaN,d,s
3,3116948,3.47,x,t,n,f,s,c,n,4.98,...,NaN,NaN,w,NaN,n,t,z,NaN,d,u
4,3116949,6.17,x,h,y,f,p,NaN,y,6.73,...,NaN,NaN,y,NaN,y,t,NaN,NaN,d,u


In [6]:
train.isnull().sum()

,0
id,0
class,0
cap-diameter,4
cap-shape,40
cap-surface,671023
cap-color,12
does-bruise-or-bleed,8
gill-attachment,523936
gill-spacing,1258435
gill-color,57


In [7]:
drop_cols = ["veil-type", "veil-color", "stem-root", "stem-surface", "spore-print-color", 'cap-surface', 'gill-attachment', 'gill-spacing', 'ring-type']

X = train.drop(columns=["id", "class"] + drop_cols)
X_test = test.drop(columns=["id"] + drop_cols)

In [8]:
# Encode target
le = LabelEncoder()
y = le.fit_transform(train["class"])  # 'e' -> 0, 'p' -> 1

In [9]:
# ===============================
# 3. Train/Validation Split
# ===============================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
# ===============================
# 4. Define Preprocessing
# ===============================
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["float64", "int64"]).columns

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_cols),
        ("numerical", numerical_transformer, numerical_cols)
    ]
)

In [14]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False])
    }

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(**params, random_state=42))
    ])

    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy")
    return scores.mean()

In [17]:
def objective(trial):

    model_name = trial.suggest_categorical(
        "model",
        ["LogisticRegression", "RandomForest"]
    )

    # ---------------- Logistic Regression ----------------
    if model_name == "LogisticRegression":
        C = trial.suggest_float("lr_C", 0.1, 100, log=True)
        solver = trial.suggest_categorical("lr_solver", ["liblinear", "lbfgs", "saga", "newton-cg"])
        penalty = None
        if solver in ["lbfgs", "newton-cg"]:
            penalty = trial.suggest_categorical("lr_penalty_lbfgs_nc", ["l2", None])
        elif solver == "liblinear":
            penalty = trial.suggest_categorical("lr_penalty_liblinear", ["l1", "l2"])
        else:  # saga
            penalty = trial.suggest_categorical("lr_penalty_saga", ["l1", "l2", "elasticnet", None])
        l1_ratio = trial.suggest_float("lr_l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None

        model = LogisticRegression(
            C=C, solver=solver, penalty=penalty, l1_ratio=l1_ratio,
            max_iter=5000, random_state=42
        )

    # ---------------- Random Forest ----------------
    elif model_name == "RandomForest":
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 50, 300),
            max_depth=trial.suggest_int("rf_max_depth", 3, 20),
            min_samples_split=trial.suggest_int("rf_min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 10),
            bootstrap=trial.suggest_categorical("rf_bootstrap", [True, False]),
            random_state=42, n_jobs=1
        )

  # ---------------- Pipeline ----------------
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # Use n_jobs=1 to avoid Windows parallelism errors
    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring="accuracy", n_jobs=1)
    return scores.mean()

In [ ]:
# ===============================
# 6. Run Optuna
# ===============================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # increase n_trials for better tuning

In [ ]:

print("Best Params:", study.best_params)
best_params = study.best_params

In [19]:
best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=57, max_depth=18, min_samples_split=6, min_samples_leaf=9, bootstrap=False))
])

In [20]:
best_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  Index(['cap-shape', 'cap-color', 'does-bruise-or-bleed', 'gill-color',
       'stem-color', 'has-ring', 'habitat', 'season'],
      dtype='object')),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  Index(['cap-diameter', 'stem-height', 'stem-width'], dtype='object'))])),
                ('model',
                 RandomForestClassifier(bootstrap=False, max_depth=18,
                                        min_samples_leaf=9, min_samples_split=6,
                                        n_estimators=57))])

In [21]:
# Validate on hold-out set
y_valid_pred = best_model.predict(X_valid)
val_acc = accuracy_score(y_valid, y_valid_pred)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.9793


In [22]:
# ===============================
# 8. Predict on Test + Submission
# ===============================
y_pred = best_model.predict(X_test)
y_pred_labels = le.inverse_transform(y_pred)  # back to 'e' / 'p'

In [23]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": y_pred_labels
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv file created with labels (e/p)")

✅ submission.csv file created with labels (e/p)


In [ ]:
# ===============================
# 7. Train Best Model
# ===============================
#best_model = Pipeline(steps=[
#    ("preprocessor", preprocessor),
#    ("model", RandomForestClassifier(**best_params, random_state=42, n_jobs=-1))
#])
